# Exploratory Data Analysis: NYC Short-Term Rental Prices

This notebook explores the raw `sample.csv` artifact produced by the `download` step of the pipeline. The goal is to understand the data well enough to decide on the basic cleaning rules that will later be moved into the `basic_cleaning` MLflow component.

**Outline**
1. Fetch the `sample.csv` artifact from the `nyc_airbnb` W&B project
2. Profile the data with ydata-profiling
3. Inspect missing values, data types and price outliers
4. Apply the basic fixes (price range filter, `last_review` to datetime)
5. Verify the fixes and finish the W&B run

## 1. Fetch the data from W&B

We start a W&B run in the `eda` group so this exploration is tracked alongside the pipeline runs. `save_code=True` uploads and versions this notebook in W&B. Using the `:latest` alias pins the exact artifact version the notebook consumed, which keeps the analysis traceable in the artifact lineage.

In [ ]:
import wandb
import pandas as pd

# Start a W&B run so this EDA is versioned together with the pipeline
run = wandb.init(project="nyc_airbnb", group="eda", save_code=True)

# Download the raw sample produced by the "download" step
local_path = wandb.use_artifact("sample.csv:latest").file()
df = pd.read_csv(local_path)

df.head()

In [ ]:
# Basic shape and column types
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
df.info()

## 2. Profile report

ydata-profiling (the maintained successor of pandas-profiling) gives a quick overview of every column: distributions, missing values, correlations and warnings.

In [ ]:
import ydata_profiling

profile = ydata_profiling.ProfileReport(df)
profile.to_notebook_iframe()

## 3. Findings from the profile

The report highlights three issues that matter for modeling.

### 3.1 Missing values
Several columns contain missing values (`name`, `host_name`, `last_review`, `reviews_per_month`). The missing values in `last_review` and `reviews_per_month` occur together, because listings that were never reviewed have neither a last review date nor a review rate.

In [ ]:
# Count and share of missing values per column (only columns with missing values)
missing = df.isna().sum()
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({"missing": missing, "percent": missing_pct}).query("missing > 0")

**Decision: we do not impute missing values here.** New data arriving in production will contain the same kinds of gaps, so imputation has to be part of the inference pipeline (the scikit-learn pipeline exported with the model). If we imputed here, the model would be trained on data that looks different from what it receives at prediction time, and the imputation logic would not be versioned with the model. The `train_random_forest` step therefore handles missing values with `SimpleImputer` steps inside the model pipeline.

### 3.2 `last_review` is stored as text
The `last_review` column is read as a string (`object`) instead of a date.

In [ ]:
# last_review is stored as a string, not as a datetime
print(df["last_review"].dtype)
df["last_review"].dropna().head()

### 3.3 Price outliers
The `price` column has a long right tail with extreme values and also contains zeros, which are not realistic nightly prices.

In [ ]:
# Summary statistics of price, including extreme quantiles
df["price"].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99])

In [ ]:
# Number of listings that fall outside the range agreed with stakeholders
print("Price == 0:", (df["price"] == 0).sum())
print("Price < 10:", (df["price"] < 10).sum())
print("Price > 350:", (df["price"] > 350).sum())

After discussing with stakeholders, the accepted nightly price range is **$10 to $350, inclusive**. Listings outside that range are treated as outliers and removed, since the business case is estimating typical prices.

## 4. Apply the basic fixes

1. Keep only rows with `10 <= price <= 350` (`Series.between` is inclusive on both ends).
2. Convert `last_review` to a datetime.

These are exactly the operations the `basic_cleaning` component reimplements, with `min_price` and `max_price` read from `config.yaml`.

In [ ]:
# Drop price outliers (inclusive range)
min_price = 10
max_price = 350
idx = df["price"].between(min_price, max_price)
df = df[idx].copy()

# Convert last_review to datetime (missing values become NaT, they are not imputed)
df["last_review"] = pd.to_datetime(df["last_review"])

## 5. Verify the fixes

In [ ]:
# Prices must now be inside the accepted range and last_review must be a datetime
df.info()
print("Price range after cleaning:", df["price"].min(), "to", df["price"].max())
assert df["price"].between(min_price, max_price).all()
assert pd.api.types.is_datetime64_any_dtype(df["last_review"])

In [ ]:
# Missing values are still present on purpose: they are handled in the inference pipeline
df.isna().sum()[lambda s: s > 0]

Prices are now within the $10 to $350 range, `last_review` is a datetime, and the missing values remain for the inference pipeline to handle.

## 6. Finish the W&B run

In [ ]:
run.finish()